# Preview and publish lightweight results

This notebook validates a Drive result bundle, previews the exact Git changes, and never stages or commits automatically.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()
LOCAL_REPOSITORY = str(REPO_DIR)
RESULT_BUNDLE_ID = os.environ.get("RESULT_BUNDLE_ID", "")
GIT_USER_NAME = os.environ.get("GIT_USER_NAME", "")
GIT_USER_EMAIL = os.environ.get("GIT_USER_EMAIL", "")


In [ ]:
print('Repository and storage initialized; no Git mutation performed.')


## Available Drive result bundles

In [ ]:
from pathlib import Path
bundles_root = Path(DRIVE_ROOT) / 'result_bundles'
available_bundles = sorted(p.name for p in bundles_root.iterdir() if p.is_dir())
available_bundles

## Dry-run preview

The command validates required files, run and checkpoint hashes, class/track compatibility, metrics, secrets, paths, extensions, and file sizes before copying anything.

In [ ]:
print('SMOKE_TEST: publishing step skipped.' if SMOKE_TEST else 'Run this publishing step only after selecting and reviewing a valid result bundle.')


## Copy approved files and inspect the diff

Run this cell only after the dry-run succeeds. It still does not stage or commit.

In [ ]:
print('SMOKE_TEST: publishing step skipped.' if SMOKE_TEST else 'Run this publishing step only after selecting and reviewing a valid result bundle.')


## Identity and safe authentication

Set identity only for this repository. Preferred authentication is a temporary Colab Secret; never print, persist, or put the token in Git configuration, notebook output, Drive, or result manifests. Committing and pushing from a local computer is the safer alternative.

In [ ]:
print('SMOKE_TEST: publishing step skipped.' if SMOKE_TEST else 'Run this publishing step only after selecting and reviewing a valid result bundle.')


## Results branch, staging, validation, commit, and pull request

Inspect remote differences before choosing a rebase or merge. Do not force-push by default. Stage only approved paths with `git add results/ benchmark_data/`, never `git add .`. Verify `gh auth status` before an optional PR command.

Suggested commands:

```bash
git fetch origin
git checkout -B experiment-results origin/main
git status
git add results/ benchmark_data/
git diff --cached --stat
python scripts/validate_results.py --repo-results results/
git commit -m "results: add validated VisDrone benchmark evaluation"
git push -u origin experiment-results
gh auth status && gh pr create --base main --head experiment-results --title "Add latest VisDrone benchmark results" --body-file results/reports/pull_request_summary.md
```

In [ ]:
!python scripts/validate_results.py --repo-results results/